<a href="https://colab.research.google.com/github/budennovsk/Pandas/blob/master/v6_2_level_model_rerank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install implicit catboost rectools[lightfm]



In [ ]:
from pathlib import Path
import typing as tp
import warnings

import pandas as pd
import numpy as np

from implicit.nearest_neighbours import CosineRecommender
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier, CatBoostRanker
try:
    from lightgbm import LGBMClassifier, LGBMRanker
    LGBM_AVAILABLE = True
except ImportError:
    warnings.warn("lightgbm is not installed. Some parts of the notebook will be skipped.")
    LGBM_AVAILABLE = False
from rectools.dataset import Interactions
from lightfm import LightFM
from rectools import Columns
from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity,MAP,NDCG,HitRate,calc_metrics
from rectools.models import (
    ImplicitALSWrapperModel,
    ImplicitBPRWrapperModel,
    LightFMWrapperModel,
    PureSVDModel,
    ImplicitItemKNNWrapperModel,
    EASEModel,
    RandomModel,
    PopularInCategoryModel,
    PopularModel,
    EASEModel, SASRecModel, BERT4RecModel)
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.nearest_neighbours import CosineRecommender
from rectools.models.base import ExternalIds
from rectools.models.ranking import (
    CandidateRankingModel,
    CandidateGenerator,
    Reranker,

    CatBoostReranker,
    CandidateFeatureCollector,
    PerUserNegativeSampler
)
from rectools.models.nn.item_net import CatFeaturesItemNet, IdEmbeddingsItemNet
from rectools.model_selection import cross_validate, TimeRangeSplitter,LastNSplitter,Splitter
from tqdm.auto import tqdm
import datetime
import dill
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from zipfile import ZipFile
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [ ]:
path_users = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/users.csv'
path_items = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/items.csv'
path_interactions = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/interactions.csv'


users_all = pd.read_csv(path_users)
items_all = pd.read_csv(path_items)
interactions = (
    pd.read_csv(path_interactions, parse_dates=["last_watch_dt"])
    .rename(columns={"last_watch_dt": Columns.Datetime})
)

# interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"], errors="coerce")
# interactions = interactions.head(1000000)
users = users_all[users_all['user_id'].isin(interactions['user_id'].unique())]
items = items_all[items_all['item_id'].isin(interactions['item_id'].unique())]
interactions["weight"] = 1
# dataset = Dataset.construct(interactions)
# RANDOM_STATE = 32


# dataset
# interactions[Columns.Datetime] = pd.to_datetime(interactions[Columns.Datetime], format='%Y-%m-%d')

In [ ]:
interactions.info()

In [ ]:
max_date = interactions["datetime"].max()
train = interactions[interactions["datetime"] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions["datetime"] >= max_date - pd.Timedelta(days=7)].copy()

print(f"train: {train.shape}")
print(f"test: {test.shape}")

In [ ]:
# отфильтруем холодных пользователей из теста
cold_users = list(set(test['user_id']) - set(train['user_id']))
len(cold_users)

In [ ]:
lfm_date_threshold = train['datetime'].quantile(q=0.6, interpolation='nearest')
lfm_date_threshold

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# предполагается, что train уже есть и train['last_watch_dt'] уже datetime64[ns]
train = train.copy()
train["datetime"] = pd.to_datetime(train["datetime"], errors="coerce")
train = train.dropna(subset=["datetime"]).copy()

# 1) Группировка по дням + количество строк
daily = (
    train.assign(day=train["datetime"].dt.floor("D"))
         .groupby("day", as_index=False)
         .size()
         .rename(columns={"size": "rows"})
         .sort_values("day")
)

# 2) (опционально) кумулятивная доля — помогает увидеть, где будет quantile(0.6)
daily["cum_rows"] = daily["rows"].cumsum()
daily["cum_share"] = daily["cum_rows"] / daily["rows"].sum()

display(daily.head())
display(daily.tail())

# 3) Порог по квантилю времени (как у тебя)
lfm_date_threshold = train["datetime"].quantile(q=0.6, interpolation="nearest")
print("lfm_date_threshold:", lfm_date_threshold)

# 4) График активности по дням
plt.figure(figsize=(14, 5))
plt.plot(daily["day"], daily["rows"], marker="o", linewidth=1)
plt.axvline(pd.to_datetime(lfm_date_threshold).floor("D"), color="red", linestyle="--", linewidth=2, label="q=0.6 threshold (day)")
plt.title("Количество взаимодействий по дням (train, datetime)")
plt.xlabel("День")
plt.ylabel("Количество строк")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 5) График кумулятивной доли (чтобы понять, где примерно 60%)
plt.figure(figsize=(14, 4))
plt.plot(daily["day"], daily["cum_share"], marker=".", linewidth=1)
plt.axhline(0.6, color="red", linestyle="--", linewidth=2, label="0.6")
plt.axvline(pd.to_datetime(lfm_date_threshold).floor("D"), color="red", linestyle="--", linewidth=2, label="threshold day")
plt.title("Кумулятивная доля строк по дням")
plt.xlabel("День")
plt.ylabel("Cum share")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
lfm_train = train[(train['datetime'] < lfm_date_threshold)]
lfm_pred = train[(train['datetime'] >= lfm_date_threshold)]

print(f"lfm_train: {lfm_train.shape}")
print(f"lfm_pred: {lfm_pred.shape}")

In [ ]:
lfm_pred = lfm_pred[lfm_pred['user_id'].isin(lfm_train['user_id'].unique())]
print(f"lfm_pred: {lfm_pred.shape}")

In [ ]:
items_lfm_train=items[items['item_id'].isin(lfm_train['item_id'])]
items_lfm_train["genre"] = items_lfm_train["genres"].str.split(",")
genre_feature = items_lfm_train[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature_train = genre_feature[genre_feature['id'].isin(train['item_id'])]
import re

col = "value"

def canon(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s*-\s*", "-", x)  # "ток - шоу" -> "ток-шоу"
    return x

# 1) нормализуем текст
genre_feature_train[col] = genre_feature_train[col].map(canon)

# 2) склейки (синонимы/варианты)
# mapping = {
#     "советские": "русские",
#     # (по написанию)
#     "токшоу": "ток-шоу",
#     "реалити": "реалити-шоу",
#     "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
#     "мультфильм": "мультфильмы",
#     "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
#     "западные мультфильмы": "мультфильмы",
#     "детские": "для детей",
#     "для самых маленьких": "для детей",    # если хочешь объединять
# }
mapping = {
    "советские": "русские",
    # (по написанию)
    "токшоу": "ток-шоу",
    "реалити": "реалити-шоу",
    "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
    "мультфильм": "мультфильмы",
    "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
    "западные мультфильмы": "мультфильмы",
    "детские": "для детей",
    "для самых маленьких": "для детей",
    "исторические":	"историческое",
    "музыка":	"музыкальные",
    "музыка":	"мюзиклы",
    "комиксы":"по комиксам",
    "спорт":"футбол",
    "комедии":	"юмор",
}

genre_feature_train[col] = genre_feature_train[col].replace(mapping)
genre_feature_train[col].nunique()

counts = (
    genre_feature_train
    .groupby('value', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

counts_lt10 = counts[counts["count"] < 10].reset_index(drop=True)

counts_lt10  # все value, которые встречаются меньше 10 раз
rare_values = set(counts_lt10["value"])

genre_feature_train["value"] = genre_feature_train["value"].where(
    ~genre_feature_train["value"].isin(rare_values),
    "other",
)
genre_feature_train[col].nunique()
items_features_train_lfm = genre_feature_train.copy()

In [ ]:
user_lfm_train=users[users['user_id'].isin(lfm_train['user_id'])]
user_lfm_train["sex"] = user_lfm_train["sex"].map({"Ж": 1, "М": 0}).fillna(-1).astype("int8")
age_category = pd.CategoricalDtype(
    categories=[
         "unknown",
        'age_18_24',
        'age_25_34',
        'age_35_44',
        'age_45_54',
        'age_55_64',
        'age_65_inf',
    ],
    ordered=True,
)
user_lfm_train["age"] = (
    user_lfm_train["age"]
    .fillna("unknown")
    .astype(age_category)
)
income_category = pd.CategoricalDtype(
    categories=[
        "unknown",          # добавили
        "income_0_20",
        "income_20_40",
        "income_40_60",
        "income_60_90",
        "income_90_150",
        "income_150_inf",
    ],
    ordered=True,
)

# если income уже строки/категории — заполним nan и приведем тип
user_lfm_train["income"] = (
    user_lfm_train["income"]
    .fillna("unknown")
    .astype(income_category)
)
user_features_frames = []
for feature in ["sex", "age", "income"]:
    feature_frame = user_lfm_train.reindex(columns=[Columns.User, feature])
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)
user_features_train = pd.concat(user_features_frames)
user_features_train_lfm=user_features_train.copy()


In [ ]:
dataset_i_u_features_lfm_train = Dataset.construct(
    interactions_df=lfm_train,
    user_features_df=user_features_train_lfm,
    cat_user_features=["sex", "age", "income"],
    item_features_df=items_features_train_lfm,
    cat_item_features=["genre", "content_type"],
)

In [ ]:
# Take few models to compare
from datetime import timedelta
RANDOM_STATE = 42
models_1_level = {
    # "popular": PopularModel(popularity="n_interactions"),
    # "popular_7d": PopularModel(period=timedelta(days=7)),
    # "random": RandomModel(random_state=RANDOM_STATE),
    # "most_raited": PopularModel(popularity="mean_weight"),
    # "pop_cat": PopularInCategoryModel(category_feature='genre', n_categories=20),
    # "cosine_knn": ImplicitItemKNNWrapperModel(CosineRecommender()),
    # 'iALS':ImplicitALSWrapperModel(
    #       AlternatingLeastSquares(
    #         factors=10,  # latent embeddings size
    #         regularization=0.1,
    #         iterations=10,
    #         alpha=50,  # confidence multiplier for non-zero entries in interactions
    #         random_state=RANDOM_STATE)),
    'LightFM':LightFMWrapperModel(
            LightFM(no_components=32,
                    loss="warp",
                    random_state=RANDOM_STATE)),

}

# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics
metrics = {
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),
    # "serendipity@10": Serendipity(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),
    "HitRate@10": HitRate(k=10)
}

K_RECS = 10

In [ ]:
results_1_level = []
K_RECOS = 10
RANDOM_STATE = 42
NUM_THREADS =-1
for model_name, model in tqdm(models_1_level.items()):
    model_quality = {'model': model_name}

    model.fit(dataset_i_u_features_lfm_train)
    candidates_level_1 = model.recommend(
        users=lfm_pred[Columns.User].unique(),
        dataset=dataset_i_u_features_lfm_train, #dataset_feature_train train_dataset
        k=K_RECOS,
        filter_viewed=True,
    )

    metric_values = calc_metrics(metrics, candidates_level_1, lfm_pred, lfm_train)

    model_quality.update(metric_values)
    results_1_level.append(model_quality)
df_quality_1_level = pd.DataFrame(results_1_level).T

df_quality_1_level.columns = df_quality_1_level.iloc[0]

df_quality_1_level.drop('model', inplace=True)
# df_quality_1_level.style.highlight_max(color='lightgreen', axis=1)
df_quality_1_level

In [ ]:
candidates_level_1['item_id'].nunique()

In [ ]:
#катбуст

In [ ]:
pos = candidates_level_1.merge(lfm_pred,
                        on=['user_id', 'item_id'],
                        how='inner')

pos['target'] = 1
print(pos.shape)
pos.head()

In [ ]:
neg = candidates_level_1.set_index(['user_id', 'item_id'])\
        .join(lfm_pred.set_index(['user_id', 'item_id']))

neg = neg[neg['watched_pct'].isnull()].reset_index()

neg = neg.sample(frac=0.07)
neg['target'] = 0

neg.shape

In [ ]:
ctb_train_users, ctb_test_users = train_test_split(lfm_pred['user_id'].unique(),
                                                  random_state=1,
                                                  test_size=0.2)

In [ ]:
# выделяем 10% под механизм early stopping
ctb_train_users, ctb_eval_users = train_test_split(ctb_train_users,
                                                  random_state=1,
                                                  test_size=0.1)

In [ ]:
select_col = ['user_id', 'item_id', 'rank', 'target']

# Catboost train
ctb_train = shuffle(
    pd.concat([
        pos[pos['user_id'].isin(ctb_train_users)],
        neg[neg['user_id'].isin(ctb_train_users)]
])[select_col]
)

# Catboost test
ctb_test = shuffle(
    pd.concat([
        pos[pos['user_id'].isin(ctb_test_users)],
        neg[neg['user_id'].isin(ctb_test_users)]
])[select_col]
)

# for early stopping
ctb_eval = shuffle(
    pd.concat([
        pos[pos['user_id'].isin(ctb_eval_users)],
        neg[neg['user_id'].isin(ctb_eval_users)]
])[select_col]
)

In [ ]:
ctb_train['target'].value_counts(normalize=True)

In [ ]:
user_col = ['user_id', 'age', 'income', 'sex', 'kids_flg']
item_col = ['item_id', 'content_type', 'countries', 'for_kids', 'age_rating', 'studios']

train_feat = ctb_train.merge(users[user_col],
                           on=['user_id'],
                           how='left')\
                        .merge(items[item_col],
                                   on=['item_id'],
                                   how='left')

eval_feat = ctb_eval.merge(users[user_col],
                           on=['user_id'],
                           how='left')\
                        .merge(items[item_col],
                                   on=['item_id'],
                                   how='left')
test_feat = ctb_test.merge(users[user_col],
                           on=['user_id'],
                           how='left')\
                    .merge(items[item_col],
                               on=['item_id'],
                               how='left')

In [ ]:
drop_col = ['user_id', 'item_id']
target_col = ['target']
cat_col = ['age', 'income', 'sex', 'content_type', 'countries', 'studios']
X_train, y_train = train_feat.drop(drop_col + target_col, axis=1), train_feat[target_col]
X_val, y_val = eval_feat.drop(drop_col + target_col, axis=1), eval_feat[target_col]
X_train, y_train = train_feat.drop(drop_col + target_col, axis=1), train_feat[target_col]
X_val, y_val = eval_feat.drop(drop_col + target_col, axis=1), eval_feat[target_col]
# fillna for catboost with the most frequent value
X_train = X_train.fillna(X_train.mode().iloc[0])
X_val = X_val.fillna(X_train.mode().iloc[0])
test_feat = test_feat.fillna(X_train.mode().iloc[0])
# X_train = X_train.fillna('unknown')
# X_val = X_val.fillna('unknown')
# test_feat = test_feat.fillna('unknown')
X_test, y_test = test_feat.drop(drop_col + target_col, axis=1), test_feat['target']

In [ ]:
from catboost import CatBoostClassifier

# параметры для обучения
est_params = {
  'subsample': 0.9,
  'max_depth': 5,
  'n_estimators': 2000,
  'learning_rate': 0.1,
  'thread_count': 20,
  'random_state': 42,
  'verbose': 200,
}

ctb_model = CatBoostClassifier(**est_params)
ctb_model.fit(X_train,
              y_train,
              eval_set=(X_val, y_val),
              early_stopping_rounds=100,
              cat_features=cat_col,
              plot=False)

In [ ]:
import shap
from catboost import Pool

# сэмплируем для shap_values
X_train_subs, _, y_train_subs, __ = train_test_split(X_train, y_train,
                                                     test_size=0.9,
                                                     random_state=42)
# считаем shap_values
shap_values = ctb_model.get_feature_importance(Pool(X_train_subs, y_train_subs,
                                                   cat_features=cat_col), type='ShapValues')

expected_value = shap_values[0, -1]
shap_values = shap_values[:, :-1]
print(X_train.shape,X_train_subs.shape)
plt.title("Важность фичей на train")

shap.summary_plot(
    shap_values,
    X_train_subs
)

In [ ]:
y_pred = ctb_model.predict_proba(X_test)
y_pred_yt = ctb_model.predict(X_test)
from sklearn.metrics import roc_auc_score

f"ROC AUC score = {roc_auc_score(y_test,y_pred[:,1],):.2f}"

In [ ]:

# оставляем только теплых пользователей
test = test[test['user_id'].isin(lfm_train['user_id'].unique())]

In [ ]:
results_lfm_test= []
K_RECOS = 200
RANDOM_STATE = 42
NUM_THREADS =-1
for model_name, model in tqdm(models_1_level.items()):
    model_quality = {'model': model_name}

    model.fit(dataset_i_u_features_lfm_train)
    candidates_lfm_test = model.recommend(
        users=test[Columns.User].unique(),
        dataset=dataset_i_u_features_lfm_train, #dataset_feature_train train_dataset
        k=K_RECOS,
        filter_viewed=True,
    )

    metric_values = calc_metrics(metrics, candidates_lfm_test, test, lfm_train)

    model_quality.update(metric_values)
    results_lfm_test.append(model_quality)
df_quality_1_level = pd.DataFrame(results_lfm_test).T

df_quality_1_level.columns = df_quality_1_level.iloc[0]

df_quality_1_level.drop('model', inplace=True)
# df_quality_1_level.style.highlight_max(color='lightgreen', axis=1)
df_quality_1_level

In [ ]:
lfm_ctb_prediction = candidates_lfm_test.copy()

# фичи для теста
score_feat = lfm_ctb_prediction.merge(users[user_col],
                                   on=['user_id'],
                                   how='left')\
                                .merge(items[item_col],
                                       on=['item_id'],
                                       how='left')

# fillna for catboost with the most frequent value
score_feat = score_feat.fillna(X_train.mode().iloc[0])
score_feat.head()

In [ ]:
# catboost predict_proba
ctb_prediction = ctb_model.predict_proba(score_feat[X_test.columns])

lfm_ctb_prediction['ctb_pred'] = ctb_prediction[:, 1]
lfm_ctb_prediction.head(3)

In [ ]:
candidates_lfm_test.copy().sort_values(
    by=['user_id'], ascending=[True]).head(10)

In [ ]:
# сортируем по скору внутри одного пользователя и проставляем новый ранг
lfm_ctb_prediction = lfm_ctb_prediction.sort_values(
    by=['user_id', 'ctb_pred'], ascending=[True, False])
lfm_ctb_prediction['rank_ctb'] = lfm_ctb_prediction.groupby('user_id').cumcount() + 1
lfm_ctb_prediction.head(10)

In [ ]:
lfm_ctb_prediction.shape,candidates_lfm_test.shape

In [ ]:
# интересно сравнить ранки 1 этапа lightfm и двухэтапной модели
pd.crosstab(lfm_ctb_prediction[lfm_ctb_prediction['rank'] <= 10]['rank'],
            lfm_ctb_prediction[lfm_ctb_prediction['rank_ctb'] <= 10]['rank_ctb'])\
    .style.background_gradient(cmap='spring')

In [ ]:
lfm_ctb_cand = (
    lfm_ctb_prediction[["user_id", "item_id", "score", "rank_ctb"]]
    .rename(columns={"rank_ctb": "rank"})
)

metric_values_lfm_pred = calc_metrics(metrics, candidates_level_1, lfm_pred, lfm_train)
metric_values_lfm_test = calc_metrics(metrics, candidates_lfm_test, test, lfm_train)
metric_values_lfm_cb_bin = calc_metrics(metrics,lfm_ctb_cand, test, lfm_train)

# 1) Собираем словари в один датафрейм (строки = разные прогоны/наборы метрик)
final_res_metrics = pd.DataFrame([
    {"stage": "lfm_pred_level_1", **metric_values_lfm_pred},
    {"stage": "lfm_test_level_1", **metric_values_lfm_test},
    {"stage": "lfm_test_level_2_ctb", **metric_values_lfm_cb_bin},
])

# 2) (опционально) сделаем stage индексом для удобства
final_res_metrics = final_res_metrics.set_index("stage")

final_res_metrics